# Project 07: Build a From-Scratch AI Agent Loop

Build a small agent that plans, calls tools, reads the results, and keeps a short memory of what it has done. The loop is plain Python with no agent framework. It uses Google Gemini, or a scripted offline demo if you have no key.

Run top to bottom. Each step matches the README.

In [ ]:
import os, subprocess
REPO = "TharunPatel20/GenAI-Engineer"   # replace with your GitHub repo, e.g. "your-name/ai-engineer-course"
PROJECT = "projects/07-agentic-ai-intro"
if REPO.startswith("<"):
    print("Repo not set. Upload the project files with the Files panel (left sidebar), into a folder named 07-agentic-ai-intro, then run os.chdir('07-agentic-ai-intro') before Step 1.")
else:
    subprocess.run(["git", "clone", "-q", f"https://github.com/{REPO}.git"], check=True)
    os.chdir(REPO.split("/")[1] + "/" + PROJECT)


## Step 1: install the pinned packages

This cell installs the exact versions from `requirements.txt`. Colab may ask you to restart the session. If it does, click **Restart session**, then run the cells again from the top (Step 0 onward).

In [ ]:
%pip install -q google-genai==2.34.0 python-dotenv==1.2.4

## Step 2: API key (Colab Secrets)

Here is how to add your key, in plain words:

1. Click the **key icon** in the left sidebar (Secrets).
2. Click **Add new secret**. Name it exactly `GEMINI_API_KEY`, and paste your key as the value.
3. Turn on **Notebook access** for that secret.

Never paste the key into a cell. No key yet? Skip this step. The offline demo runs without a key.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception as e:
    print("No Colab secret found. The offline demo still runs. Details:", e)

print("key found:", bool(os.environ.get("GEMINI_API_KEY", "").strip()))

### Build Step 1: Load the key
**What:** Read `GEMINI_API_KEY` from `.env`, the same way project 04 does.
**Why:** Keys belong in a file you do not share, not in your code.

In Colab the key comes from Colab Secrets (Step 2 above). This cell also reads a `.env` file in the current folder if one exists. If no key is set, the offline demo runs instead of stopping.

In [ ]:
from pathlib import Path

from dotenv import load_dotenv

# Colab has no __file__, so this reads a .env file in the current folder if one exists.
load_dotenv(Path.cwd() / ".env")
MODEL_NAME = "gemini-2.5-flash"
MAX_STEPS = 6

### Build Step 2: Write the tools
**What:** Three plain functions: `add_numbers`, `get_word_count`, and `lookup_price` (a fake price list). The `TOOLS` dict maps each name to its function.
**Why:** A tool is just a function the agent may ask to run. The docstring is what the model reads to decide when to use it. `describe_tools()` builds that list automatically.

Code copied from `tools.py`.

In [ ]:
import inspect

# A small fake price list. No real shop is contacted.
PRICES = {"laptop": 999.0, "phone": 499.0, "headphones": 89.5}


def add_numbers(a, b):
    """Add two numbers and return the sum."""
    return float(a) + float(b)


def get_word_count(text):
    """Count the words in a piece of text and return the number."""
    return len(str(text).split())


def lookup_price(item):
    """Look up the price of an item in dollars. Known items: laptop, phone, headphones."""
    key = str(item).strip().lower()
    if key not in PRICES:
        raise ValueError(f"no price for '{item}'")
    return PRICES[key]


TOOLS = {
    "add_numbers": add_numbers,
    "get_word_count": get_word_count,
    "lookup_price": lookup_price,
}


def describe_tools():
    """Return one line per tool: its name, its arguments, and its docstring."""
    lines = []
    for name, func in TOOLS.items():
        lines.append(f"- {name}{inspect.signature(func)}: {func.__doc__}")
    return "\n".join(lines)

### Build Step 3: Write the prompt
**What:** The prompt lists the tools, asks for exactly one JSON object, and shows the two allowed forms: `{"action": ..., "args": {...}}` or `{"final_answer": ...}`. It also includes the goal and the memory.
**Why:** A structured reply is easy for code to read. Free text is not.

Code copied from `agent_loop.py`. It uses `describe_tools()` from Build Step 2.

In [ ]:
# Step 3: The prompt. The model must answer with one JSON object.
PROMPT_TEMPLATE = """You are an agent that reaches a goal one step at a time.

Tools you can use:
{tools}

Reply with exactly one JSON object and nothing else. Use one of these two forms:
{{"action": "<tool name>", "args": {{"<argument name>": <value>}}}}
{{"final_answer": "<your answer>"}}

Use a tool when you need a fact or a calculation. Give the final answer only when you have every result you need.

Goal: {goal}

Steps so far (your memory):
{memory}

Your next reply (JSON only):"""


def build_prompt(goal, memory):
    memory_text = "\n".join(memory) if memory else "(none yet)"
    return PROMPT_TEMPLATE.format(tools=describe_tools(), goal=goal, memory=memory_text)

### Build Step 4: Parse the reply
**What:** Find the JSON object in the model's text and load it. Models sometimes wrap JSON in code fences, so the parser removes those first.
**Why:** The model is not always perfect. Parsing must fail safely.

In [ ]:
import json
import re


# Step 4: Parse the reply. Models sometimes wrap JSON in ``` fences, so remove those first.
def parse_reply(text):
    cleaned = re.sub(r"```(?:json)?", "", text).strip()
    start = cleaned.find("{")
    end = cleaned.rfind("}")
    if start == -1 or end == -1:
        raise ValueError("no JSON object found")
    reply = json.loads(cleaned[start : end + 1])
    if not isinstance(reply, dict):
        raise ValueError("reply is not a JSON object")
    return reply

### Build Step 5: Run the loop (plan, act, observe)
**What:** On each step the code asks the model for the next move. If the reply is a tool call, it runs the tool and stores the result as an observation. The observation is appended to memory, so the next prompt includes it. The loop stops at a `final_answer` or after `MAX_STEPS`.
**Why:** This is the whole idea of an agent. Everything else is detail.

This cell only defines the loop. Build Step 6 gives it a model to call.

In [ ]:
GOAL = (
    "Find the price of a laptop, add 12 and 30, count the words in "
    "'plan act observe', and report all three results."
)


# Step 5: The loop. Plan (ask the model), act (run a tool), observe (store the result).
def run_agent(goal, ask_model):
    memory = []  # one line per past step; it is added to every prompt
    for step in range(1, MAX_STEPS + 1):
        prompt = build_prompt(goal, memory)
        try:
            reply_text = ask_model(prompt, step)
        except Exception as error:  # for example a network or quota error
            print(f"Model call failed: {error}")
            return None
        print(f"\nStep {step} reply: {reply_text.strip()}")

        try:
            reply = parse_reply(reply_text)
        except (ValueError, json.JSONDecodeError) as error:
            observation = f"Error: your reply was not valid JSON ({error}). Reply with one JSON object."
            memory.append(f"Step {step}: bad reply -> {observation}")
            print(f"  Observation: {observation}")
            continue

        if "final_answer" in reply:
            return reply["final_answer"]

        action = reply.get("action")
        args = reply.get("args", {})
        if not isinstance(action, str) or action not in TOOLS:
            observation = f"Error: unknown tool '{action}'. Use one of: {', '.join(TOOLS)}."
        else:
            try:
                observation = f"Result: {TOOLS[action](**args)}"
            except Exception as error:  # the tool failed, so tell the model why
                observation = f"Error: {type(error).__name__}: {error}"

        memory.append(f"Step {step}: action={action} args={args} -> {observation}")
        print(f"  Observation: {observation}")

    return None

### Build Step 6: Offline stub and the Gemini call
**What:** `offline_model` returns the next line of `OFFLINE_SCRIPT`. `gemini_reply` calls Gemini with the same `generate_content` pattern as project 04. The next two cells choose between them: the offline demo always works, and live Gemini runs only if a key is set.
**Why:** The stub lets you run the whole loop with no account. It is labeled **OFFLINE DEMO** everywhere so you never mistake it for real model output.

This cell defines the stub and the Gemini call. It does not run anything yet.

In [ ]:
from google import genai

OFFLINE_NOTE = "OFFLINE DEMO: scripted replies, not real Gemini output"

# Replies the offline stub returns, one per step. They are fixed, not model output.
OFFLINE_SCRIPT = [
    '{"action": "lookup_price", "args": {"item": "laptop"}}',
    '{"action": "add_numbers", "args": {"a": 12, "b": 30}}',
    '{"action": "get_word_count", "args": {"text": "plan act observe"}}',
    '{"final_answer": "The laptop costs 999.0 dollars, 12 + 30 is 42.0, and the phrase has 3 words."}',
]


# Offline stub: returns the scripted reply for this step.
def offline_model(prompt, step):
    index = step - 1
    if index < len(OFFLINE_SCRIPT):
        return OFFLINE_SCRIPT[index]
    return '{"final_answer": "(script ended)"}'


# The Gemini call. Same pattern as project 04.
def gemini_reply(client, prompt):
    response = client.models.generate_content(model=MODEL_NAME, contents=prompt)
    return response.text or ""

#### Offline demo (no key needed)

This cell runs the whole loop with the scripted replies. The output is labeled **OFFLINE DEMO** and is not real Gemini output.

In [ ]:
print(f"Mode: {OFFLINE_NOTE}")
print(f"Goal: {GOAL}")
answer = run_agent(GOAL, offline_model)
print()
if answer is None:
    print(f"No final answer after {MAX_STEPS} steps, or a model call failed.")
else:
    print(f"Final answer: {answer}")
print(OFFLINE_NOTE + ". The answer is scripted, not produced by a model.")

#### Live Gemini mode (only if you added a key in Step 2)

If no key is set, this cell prints a note and does nothing. Live replies will vary from run to run.

In [ ]:
import os

api_key = os.getenv("GEMINI_API_KEY", "").strip()
if not api_key or api_key == "your-key-here":
    print("No Gemini API key found, so the live run is skipped. Use the offline demo above.")
else:
    client = genai.Client(api_key=api_key)
    print(f"Model: {MODEL_NAME}")
    print(f"Goal: {GOAL}")
    answer = run_agent(GOAL, lambda prompt, step: gemini_reply(client, prompt))
    print()
    if answer is None:
        print(f"No final answer after {MAX_STEPS} steps, or a model call failed.")
    else:
        print(f"Final answer: {answer}")

## Checkpoints and what you learned

- After Build Step 2: in a new cell, run `print(describe_tools())`. It lists the three tools with their arguments and docstrings
- After Build Step 5: each step prints a reply and an observation, and memory grows by one line per step
- After Build Step 6 (offline demo): the `Mode: OFFLINE DEMO` line appears, and the run ends with a final answer in 4 steps
- After Build Step 6 (live): `Model: gemini-2.5-flash` appears, and the agent calls the same tools for its own plan

You learned the plan, act, observe loop and why memory is needed for multi-step work, plain Python functions as tools with docstrings as their descriptions, asking an LLM for JSON and parsing it safely, max-step limits and error observations, and single-agent versus multi-agent designs and the trade-offs of frameworks.

To keep going, try the stretch goals in the project README.

Run locally instead? See the README setup section.